In [0]:
# ============================================================
# 1_topic_tagging_v5_classify - Round 3
# LLM 배치 분류 파이프라인 (v1 결합 없음, LLM 결과만 저장)
# ============================================================
# 흐름:
#  1. GROUP_STATUS_TABLE is_done=0 그룹 로드
#  2. 그룹별 샘플링 → LLM 분류
#  3. DETAIL_TABLE(회차별) 저장 + GROUP_STATUS_TABLE is_done=1
#  4. 파이프라인 완료 후 V1_DETAIL_TABLE에 머지
#  5. SUMMARY_TABLE 생성
# ============================================================
# v4 변경사항:
#  - 200자 초과 메모 → LLM 1문장 요약 후 요약 메모로 분류
#  - DETAIL_TABLE에 memo_summary 컨럼 추가 (요약된 문장)
#  - Integration 테이블은 기존 컨럼 포맷 유지 (memo_summary 미포함)
#  - 버전 추적용 테이블 추가
#  - 우선 국가/브랜드 기반 샘플링 전략 적용
#  - 샘플링 전략: 그룹당 최대 1000건 고정
# ============================================================

from __future__ import annotations

import json
import re
import time
from typing import Any, Dict, List, Optional

import pandas as pd
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

# --------------------------------------------------
# LLM Endpoint
# --------------------------------------------------
ENDPOINT = "databricks-gpt-5-4-mini"

# --------------------------------------------------
# DB / Tables
# --------------------------------------------------
PROMPT_VERSION = "_v2"
SAVE_DB = "sandbox.z_jungryo_lee"

# --------------------------------------------------
# 샘플링 회차 설정 (4차 샘플링)
# --------------------------------------------------
SAMPLING_ROUND = 7

SOURCE_TABLE = "kic_data_ods.buzzmetrix.buzzmetrix"

RULE_PROFILE_TABLE  = f"{SAVE_DB}.category_topic_rule_profile"
TOPIC_POOL_TABLE    = f"{SAVE_DB}.category_topic_catalog"

# 회차별 DETAIL 테이블 (round 번호 포함)
DETAIL_TABLE        = f"{SAVE_DB}.category_topic_detail{PROMPT_VERSION}_round{SAMPLING_ROUND}"
SUMMARY_TABLE       = f"{SAVE_DB}.category_topic_summary{PROMPT_VERSION}_round{SAMPLING_ROUND}"
PROGRESS_TABLE      = f"{SAVE_DB}.category_topic_progress{PROMPT_VERSION}"
FAILED_GROUP_TABLE  = f"{SAVE_DB}.category_topic_failed_group{PROMPT_VERSION}"
GROUP_STATUS_TABLE  = f"{SAVE_DB}.category_topic_group_status"

# 이전 회차 DETAIL 테이블 목록 (중복 샘플링 방지용)
PREV_DETAIL_TABLES = [
    f"{SAVE_DB}.category_topic_detail{PROMPT_VERSION}_round{i}"
    for i in range(2, SAMPLING_ROUND)
]
# Round 1 원본 테이블 (round suffix 없이 저장된 것) 추가
PREV_DETAIL_TABLES.append(f"{SAVE_DB}.category_topic_detail{PROMPT_VERSION}")

# 통합 결과 테이블 (merge 대상)
V1_DETAIL_TABLE = f"{SAVE_DB}.category_topic_detail_integration"

# 버전 추적 테이블
VERSION_TRACKING_TABLE = f"{SAVE_DB}.category_topic_version_log"

# --------------------------------------------------
# Thresholds / Limits
# --------------------------------------------------
MAX_SAMPLE_ROWS      = 1000
CLASSIFY_BATCH_SIZE  = 25
MIN_ROWS_PER_TOPIC   = 10
MAX_TOKENS           = 2200
MAX_RETRIES          = 3
BACKOFF_BASE         = 1.8
RATE_LIMIT_SECONDS   = 0.4

# --------------------------------------------------
# 메모 요약 임계값 (v4 신규)
# --------------------------------------------------
MEMO_SUMMARIZE_THRESHOLD = 200  # 이 길이 초과 시 1문장 요약 후 분류
SUMMARIZE_BATCH_SIZE     = 20   # 요약 LLM 배치 크기

# --------------------------------------------------
# 샘플링 상수 (단순화: 그룹당 최대 1000건)
# --------------------------------------------------
SAMPLE_CAP = 1000  # 그룹당 최대 샘플 수 (고정)

# --------------------------------------------------
# 소스 데이터 필터 조건
# --------------------------------------------------
MIN_YEAR = 2022  # year >= 2022 데이터만 샘플링

# 브랜드 필터 (5개 브랜드만 대상)
TARGET_BRANDS = ["Samsung", "LG", "SONY", "TCL", "HISENSE"]

# 우선 샘플링 국가 (이 국가들을 먼저 샘플링, 나머지는 후순위)
PRIORITY_COUNTRIES = [
    "Brazil", "Germany", "India", "Korea",
    "United Kingdom", "United States", "Vietnam"
]

print("[CONFIG] Loaded - Round 4 (v4: 메모 요약 + 우선국가 샘플링)")
print(f"  SAMPLING_ROUND           = {SAMPLING_ROUND}")
print(f"  ENDPOINT                 = {ENDPOINT}")
print(f"  SOURCE_TABLE             = {SOURCE_TABLE}")
print(f"  DETAIL_TABLE             = {DETAIL_TABLE}")
print(f"  PREV_DETAIL_TABLES       = {PREV_DETAIL_TABLES}")
print(f"  V1_DETAIL_TABLE          = {V1_DETAIL_TABLE}")
print(f"  GROUP_STATUS_TABLE       = {GROUP_STATUS_TABLE}")
print(f"  VERSION_TRACKING_TABLE   = {VERSION_TRACKING_TABLE}")
print(f"  MEMO_SUMMARIZE_THRESHOLD = {MEMO_SUMMARIZE_THRESHOLD}")
print(f"  SAMPLE_CAP               = {SAMPLE_CAP} (그룹당 최대 고정)")
print(f"  MIN_YEAR                 = {MIN_YEAR}")
print(f"  TARGET_BRANDS            = {TARGET_BRANDS}")
print(f"  PRIORITY_COUNTRIES       = {PRIORITY_COUNTRIES}")

In [0]:
# ============================================================
# 2) Helpers & Checkpoint
# ============================================================

def clean_text(x: Any) -> str:
    return "" if x is None else re.sub(r"\s+", " ", str(x)).strip()


def extract_json(text: str) -> Dict[str, Any]:
    text = clean_text(text)
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r"\{.*\}", text, flags=re.S)
    if match:
        candidate = match.group(0)
        try:
            return json.loads(candidate)
        except Exception:
            candidate = re.sub(r",\s*}", "}", candidate)
            candidate = re.sub(r",\s*]", "]", candidate)
            return json.loads(candidate)
    raise ValueError(f"Invalid JSON: {text[:1000]}")


def chunk_list(items: List[Any], size: int) -> List[List[Any]]:
    return [items[i:i + size] for i in range(0, len(items), size)]


def sc_label(sc_measurement: int) -> str:
    return {1: "긍정", -1: "부정"}.get(sc_measurement, "기타")


def overall_label(sc_measurement: int) -> str:
    return {1: "전반적 긍정", -1: "전반적 부정"}.get(sc_measurement, "전반적 평가")


def call_llm(messages: List[Dict[str, str]], max_tokens: int = MAX_TOKENS) -> Dict[str, Any]:
    from mlflow.deployments import get_deploy_client
    client = get_deploy_client("databricks")
    payload = {"messages": messages, "temperature": 0.0, "max_tokens": max_tokens}
    last_error: Optional[Exception] = None
    for attempt in range(MAX_RETRIES):
        resp = None
        try:
            resp = client.predict(endpoint=ENDPOINT, inputs=payload)
            if isinstance(resp, dict):
                if "choices" in resp and resp["choices"]:
                    return extract_json(resp["choices"][0]["message"]["content"])
                if "predictions" in resp and resp["predictions"]:
                    pred0 = resp["predictions"][0]
                    if isinstance(pred0, dict) and "content" in pred0:
                        return extract_json(pred0["content"])
                    if isinstance(pred0, str):
                        return extract_json(pred0)
                if "content" in resp:
                    return extract_json(resp["content"])
            if isinstance(resp, str):
                return extract_json(resp)
            raise ValueError(f"Unexpected response schema: {resp}")
        except Exception as exc:
            last_error = exc
            print(f"[LLM ERROR] attempt={attempt + 1}/{MAX_RETRIES}, error={repr(exc)}")
            time.sleep(BACKOFF_BASE ** attempt)
    raise RuntimeError(f"LLM call failed: {repr(last_error)}")


# --------------------------------------------------
# Table I/O Helpers
# --------------------------------------------------
def append_table(df, table_name: str) -> None:
    if spark.catalog.tableExists(table_name):
        df.write.mode("append").format("delta").saveAsTable(table_name)
    else:
        df.write.mode("overwrite").format("delta").saveAsTable(table_name)


def delete_group_rows(table_name, cate_1_depth, cate_2_depth, sc_measurement):
    """해당 그룹 행 삭제."""
    if not spark.catalog.tableExists(table_name):
        return
    spark.sql(f"""
        DELETE FROM {table_name}
        WHERE cate_1_depth = '{cate_1_depth}'
          AND cate_2_depth = '{cate_2_depth}'
          AND sc_measurement = {sc_measurement}
    """)


def update_group_status(c1, c2, sc, is_done=1):
    """GROUP_STATUS_TABLE is_done 업데이트."""
    spark.sql(f"""
        UPDATE {GROUP_STATUS_TABLE}
        SET is_done = {is_done}
        WHERE cate_1_depth = '{c1}'
          AND cate_2_depth = '{c2}'
          AND sc_measurement = {sc}
    """)


# --------------------------------------------------
# Progress Logging
# --------------------------------------------------
PROGRESS_STRUCT = T.StructType([
    T.StructField("cate_1_depth", T.StringType(), True),
    T.StructField("cate_2_depth", T.StringType(), True),
    T.StructField("sc_measurement", T.IntegerType(), True),
    T.StructField("stage", T.StringType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("message", T.StringType(), True),
    T.StructField("event_ts", T.TimestampType(), True),
])

FAILED_STRUCT = T.StructType([
    T.StructField("cate_1_depth", T.StringType(), True),
    T.StructField("cate_2_depth", T.StringType(), True),
    T.StructField("sc_measurement", T.IntegerType(), True),
    T.StructField("stage", T.StringType(), True),
    T.StructField("error_message", T.StringType(), True),
    T.StructField("event_ts", T.TimestampType(), True),
])


def log_progress(cate_1_depth, cate_2_depth, sc_measurement, stage, status, message=""):
    row = [(
        clean_text(cate_1_depth), clean_text(cate_2_depth), int(sc_measurement),
        clean_text(stage), clean_text(status), clean_text(message),
        pd.Timestamp.now().to_pydatetime(),
    )]
    append_table(spark.createDataFrame(row, schema=PROGRESS_STRUCT), PROGRESS_TABLE)


def log_failure(cate_1_depth, cate_2_depth, sc_measurement, stage, error_message):
    row = [(
        clean_text(cate_1_depth), clean_text(cate_2_depth), int(sc_measurement),
        clean_text(stage), clean_text(error_message),
        pd.Timestamp.now().to_pydatetime(),
    )]
    append_table(spark.createDataFrame(row, schema=FAILED_STRUCT), FAILED_GROUP_TABLE)


# --------------------------------------------------
# v1 결과 로드
# --------------------------------------------------
DETAIL_COLUMNS = ["_row_id", "cate_1_depth", "cate_2_depth", "sc_measurement", "memo", "topic", "description"]

def load_v1_group_results(c1: str, c2: str, sc: int):
    if not spark.catalog.tableExists(V1_DETAIL_TABLE):
        return spark.createDataFrame([], schema=T.StructType([
            T.StructField(c, T.StringType(), True) for c in DETAIL_COLUMNS
        ]))
    return spark.sql(f"""
        SELECT {', '.join(DETAIL_COLUMNS)}
        FROM {V1_DETAIL_TABLE}
        WHERE cate_1_depth = '{c1}'
          AND cate_2_depth = '{c2}'
          AND sc_measurement = {sc}
          AND _row_id IS NOT NULL
    """)


# --------------------------------------------------
# 샘플링 helpers (기분류 중복 방지 + 브랜드/국가 우선 샘플링)
#   - brand_name IN TARGET_BRANDS 필터
#   - year >= MIN_YEAR 필터
#   - PRIORITY_COUNTRIES 우선 샘플링 → 나머지 후순위
#   - 그룹당 최대 SAMPLE_CAP(1000)건 고정
# --------------------------------------------------
def _brand_filter_sql() -> str:
    """TARGET_BRANDS SQL IN 절 생성."""
    brands_str = ", ".join([f"'{b}'" for b in TARGET_BRANDS])
    return f"brand_name IN ({brands_str})"


def _get_new_memos_df(cate_1_depth: str, cate_2_depth: str, sc_measurement: int, country_filter: str = ""):
    """소스 테이블에서 기분류 메모를 제외한 신규 메모 DF.
    - brand_name IN TARGET_BRANDS
    - year >= MIN_YEAR
    - country_filter: 추가 국가 필터 조건 (우선/비우선 구분)
    """
    country_clause = f"AND ({country_filter})" if country_filter else ""
    
    source_df = spark.sql(f"""
        SELECT DISTINCT memo
        FROM {SOURCE_TABLE}
        WHERE cate_1_depth = '{cate_1_depth}'
          AND cate_2_depth = '{cate_2_depth}'
          AND sc_measurement = {sc_measurement}
          AND memo IS NOT NULL
          AND LENGTH(TRIM(memo)) > 0
          AND CAST(year AS INT) >= {MIN_YEAR}
          AND {_brand_filter_sql()}
          {country_clause}
    """)

    # 이전 회차 DETAIL_TABLE 기분류 제외
    for prev_table in PREV_DETAIL_TABLES:
        if spark.catalog.tableExists(prev_table):
            classified_df = spark.sql(f"""
                SELECT DISTINCT memo FROM {prev_table}
                WHERE cate_1_depth = '{cate_1_depth}'
                  AND cate_2_depth = '{cate_2_depth}'
                  AND sc_measurement = {sc_measurement}
            """)
            source_df = source_df.join(classified_df, on="memo", how="left_anti")

    # 현재 회차 DETAIL_TABLE 기분류 제외 (재시작 시 중복 방지)
    if spark.catalog.tableExists(DETAIL_TABLE):
        classified_df = spark.sql(f"""
            SELECT DISTINCT memo FROM {DETAIL_TABLE}
            WHERE cate_1_depth = '{cate_1_depth}'
              AND cate_2_depth = '{cate_2_depth}'
              AND sc_measurement = {sc_measurement}
        """)
        source_df = source_df.join(classified_df, on="memo", how="left_anti")

    # v1 기분류 제외
    if spark.catalog.tableExists(V1_DETAIL_TABLE):
        v1_df = spark.sql(f"""
            SELECT DISTINCT memo FROM {V1_DETAIL_TABLE}
            WHERE cate_1_depth = '{cate_1_depth}'
              AND cate_2_depth = '{cate_2_depth}'
              AND sc_measurement = {sc_measurement}
              AND _row_id IS NOT NULL
        """)
        source_df = source_df.join(v1_df, on="memo", how="left_anti")

    return source_df


def get_group_new_count(c1: str, c2: str, sc: int) -> int:
    """전체 신규 메모 수 (우선 + 비우선 국가 합산)."""
    return _get_new_memos_df(c1, c2, sc).count()


def get_group_total_count(c1: str, c2: str, sc: int) -> int:
    """year >= MIN_YEAR + brand 필터 적용된 전체 메모 수."""
    return spark.sql(f"""
        SELECT count(distinct memo) as cnt FROM {SOURCE_TABLE}
        WHERE cate_1_depth='{c1}' AND cate_2_depth='{c2}' AND sc_measurement={sc}
          AND memo IS NOT NULL AND LENGTH(TRIM(memo))>0
          AND CAST(year AS INT) >= {MIN_YEAR}
          AND {_brand_filter_sql()}
    """).collect()[0]["cnt"]


def get_dynamic_sample_size(total_count: int) -> int:
    """그룹당 최대 SAMPLE_CAP(1000)건. 남은 메모가 적으면 전수."""
    return min(total_count, SAMPLE_CAP)


def _sample_from_df(df, max_rows: int, c1: str, c2: str, sc: int):
    """결정적 샘플링 (md5 hash 기반 순서)."""
    return (
        df
        .withColumn("_rn", F.row_number().over(
            Window.orderBy(F.md5(F.concat(
                F.coalesce(F.col("memo"), F.lit("")),
                F.lit(f"||{c1}||{c2}||{sc}||seed_20260420_round{SAMPLING_ROUND}")
            )))))
        .where(F.col("_rn") <= max_rows)
        .drop("_rn")
    )


def load_group_sample_df(c1, c2, sc, max_rows):
    """우선 국가 먼저 샘플링, 남은 슬롯은 비우선 국가에서 채움.
    1) PRIORITY_COUNTRIES에서 최대 max_rows건 샘플링
    2) 부족하면 나머지 국가에서 채움
    """
    priority_str = ", ".join([f"'{c}'" for c in PRIORITY_COUNTRIES])
    
    # 1) 우선 국가 신규 메모
    priority_df = _get_new_memos_df(c1, c2, sc, country_filter=f"country IN ({priority_str})")
    priority_count = priority_df.count()
    
    if priority_count >= max_rows:
        # 우선 국가만으로 충분
        print(f"    [샘플링] 우선국가 {priority_count:,}건 ≥ {max_rows} → 우선국가만 샘플링")
        sampled = _sample_from_df(priority_df, max_rows, c1, c2, sc)
    else:
        # 우선 국가 전체 + 비우선 국가에서 나머지 채움
        remaining_slots = max_rows - priority_count
        non_priority_df = _get_new_memos_df(c1, c2, sc, country_filter=f"country NOT IN ({priority_str})")
        non_priority_count = non_priority_df.count()
        print(f"    [샘플링] 우선국가 {priority_count:,}건 + 비우선 {non_priority_count:,}건, 나머지 {remaining_slots}건 채움")
        
        # 비우선 국가에서 remaining_slots만큼 샘플링
        non_priority_sampled = _sample_from_df(non_priority_df, remaining_slots, c1, c2, sc)
        
        # 우선 전체 + 비우선 샘플 합치기
        sampled = priority_df.unionByName(non_priority_sampled)
    
    return (
        sampled
        .withColumn("_row_id", F.monotonically_increasing_id())
    ).cache()


def load_group_all_memos_df(c1: str, c2: str, sc: int):
    """전체 신규 메모 DF (전반적 그룹용). brand + year 필터 적용."""
    return _get_new_memos_df(c1, c2, sc)


print(f"[OK] Helpers loaded (v4: 우선국가 샘플링 + 브랜드 필터, round={SAMPLING_ROUND})")
print(f"     제외 대상 테이블: {PREV_DETAIL_TABLES + [V1_DETAIL_TABLE]}")
print(f"     연도 필터: year >= {MIN_YEAR}")
print(f"     브랜드 필터: {TARGET_BRANDS}")
print(f"     우선 국가: {PRIORITY_COUNTRIES}")
print(f"     샘플링: 그룹당 최대 {SAMPLE_CAP}건 (고정)")

In [0]:
# ============================================================
# 2-1) Memo Summarization Helper (v4 신규)
#      - 200자 초과 메모를 1문장으로 요약
#      - 요약된 메모로 주제분류 수행
#      - DETAIL_TABLE에는 memo_summary 컬럼 포함
#      - Integration 테이블에는 memo_summary 미포함
# ============================================================

def build_summarize_messages(batch_rows: List[Dict[str, Any]]) -> List[Dict[str, str]]:
    """메모 배치 요약용 LLM 메시지 생성.
    각 메모를 핵심 내용만 담은 1문장(한국어)으로 요약.
    """
    system = """You are a Korean text summarizer for TV product reviews.

Task: Summarize each memo into exactly ONE concise Korean sentence.
Rules:
- Keep the core opinion/evaluation/complaint intact.
- Remove unnecessary repetition, filler words, and irrelevant details.
- The summary must preserve the sentiment (positive/negative) and the main subject.
- Output in Korean only.
- Do NOT add any interpretation or opinion not present in the original.

Return only JSON:
{
  "results": [
    {
      "row_id": "",
      "summary": ""
    }
  ]
}
"""
    user = (
        "Memos to summarize:\n"
        + json.dumps(
            [{"row_id": str(r["_row_id"]), "memo": clean_text(r["memo"])} for r in batch_rows],
            ensure_ascii=False,
        )
    )
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


def summarize_memos_batch(rows: List[Dict[str, Any]]) -> Dict[str, str]:
    """
    200자 초과 메모들을 배치로 요약.
    Returns: {row_id_str: summary_text}
    """
    # 200자 초과 필터링
    long_rows = [r for r in rows if len(clean_text(r["memo"])) > MEMO_SUMMARIZE_THRESHOLD]
    
    if not long_rows:
        return {}
    
    print(f"    [요약] {len(long_rows)}/{len(rows)}건 메모 요약 시작 (>{MEMO_SUMMARIZE_THRESHOLD}자)")
    
    summary_map = {}  # row_id_str -> summary
    total_batches = (len(long_rows) + SUMMARIZE_BATCH_SIZE - 1) // SUMMARIZE_BATCH_SIZE
    
    for batch_no, batch in enumerate(chunk_list(long_rows, SUMMARIZE_BATCH_SIZE), start=1):
        try:
            result = call_llm(build_summarize_messages(batch), max_tokens=MAX_TOKENS)
            for item in result.get("results", []):
                if isinstance(item, dict) and item.get("row_id") and item.get("summary"):
                    summary_map[str(item["row_id"])] = clean_text(item["summary"])
            print(f"    [요약 BATCH] {batch_no}/{total_batches} 완료 ({len(summary_map)}건 누적)")
        except Exception as exc:
            print(f"    [요약 ERROR] batch={batch_no}, error={repr(exc)}")
            # 요약 실패 시 원본 메모 그대로 사용 (fallback)
            for r in batch:
                summary_map[str(r["_row_id"])] = clean_text(r["memo"])[:200]
        time.sleep(RATE_LIMIT_SECONDS)
    
    print(f"    [요약 완료] 총 {len(summary_map)}건 요약")
    return summary_map


def prepare_rows_for_classification(rows: List[Dict[str, Any]]) -> tuple:
    """
    분류 전 메모 전처리:
    - 200자 초과 메모 → 요약 후 요약본으로 분류
    - 200자 이하 메모 → 원본 그대로 분류
    
    Returns:
        (classify_rows, summary_map)
        - classify_rows: 분류에 사용할 rows (요약된 메모로 대체된 버전)
        - summary_map: {row_id_str: summary_text} (요약 적용된 row들만)
    """
    # 요약 수행
    summary_map = summarize_memos_batch(rows)
    
    # 분류용 rows 준비 (요약된 메모로 대체)
    classify_rows = []
    for r in rows:
        row_id_str = str(r["_row_id"])
        new_row = dict(r)  # 복사
        if row_id_str in summary_map:
            # 요약된 메모로 분류 수행
            new_row["memo"] = summary_map[row_id_str]
        classify_rows.append(new_row)
    
    return classify_rows, summary_map


print("[OK] Memo Summarization Helper 로드 완료")
print(f"     MEMO_SUMMARIZE_THRESHOLD = {MEMO_SUMMARIZE_THRESHOLD}자")
print(f"     SUMMARIZE_BATCH_SIZE = {SUMMARIZE_BATCH_SIZE}")

In [0]:
# ============================================================
# 3) Reset & Load Pending Groups
#    - Round 2: LLM 대상 그룹만 is_done=0으로 리셋
#    - 비대상 제외: cate_1_depth에 '***' 포함, sc_measurement = 0
#    - 미시작 제외: rule_profile / topic_pool 없는 그룹
#    - 리셋 후 is_done=0 그룹 로드
# ============================================================

# --------------------------------------------------
# A. Reset is_done for Round 2
# --------------------------------------------------
print(f"[BEFORE] GROUP_STATUS_TABLE")
display(
    spark.table(GROUP_STATUS_TABLE)
    .groupBy("is_done")
    .count()
    .orderBy("is_done")
)

# Step 1: 전체 is_done = 1 (우선 모두 SKIP 상태로)
spark.sql(f"UPDATE {GROUP_STATUS_TABLE} SET is_done = 1")
print("\n[Step 1] 전체 is_done = 1 설정 완료")

# Step 2: LLM 대상 그룹만 is_done = 0 으로 리셋
spark.sql(f"""
    UPDATE {GROUP_STATUS_TABLE} gs
    SET gs.is_done = 0
    WHERE gs.cate_1_depth NOT LIKE '%***%'
      AND gs.sc_measurement != 0
      AND EXISTS (
          SELECT 1 FROM {RULE_PROFILE_TABLE} rp
          WHERE rp.cate_1_depth = gs.cate_1_depth
            AND rp.cate_2_depth = gs.cate_2_depth
            AND rp.sc_measurement = gs.sc_measurement
      )
      AND EXISTS (
          SELECT 1 FROM {TOPIC_POOL_TABLE} tp
          WHERE tp.cate_1_depth = gs.cate_1_depth
            AND tp.cate_2_depth = gs.cate_2_depth
            AND tp.sc_measurement = gs.sc_measurement
      )
""")

print(f"\n[Step 2] LLM 대상 그룹만 is_done = 0 리셋 완료")
display(
    spark.table(GROUP_STATUS_TABLE)
    .groupBy("is_done")
    .count()
    .orderBy("is_done")
)

# 제외된 그룹 상세
skipped_df = (
    spark.table(GROUP_STATUS_TABLE)
    .where("is_done = 1")
    .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement")
)
skipped_count = skipped_df.count()
print(f"\n[제외된 그룹 ({skipped_count}개)] 비대상 + 미시작 (rule/topic pool 없음)")
if skipped_count > 0:
    display(skipped_df)

# --------------------------------------------------
# B. Load Pending Groups
# --------------------------------------------------
target_groups = (
    spark.table(GROUP_STATUS_TABLE)
    .where("is_done = 0")
    .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement")
    .collect()
)

total_groups_all = spark.table(GROUP_STATUS_TABLE).count()
done_groups = spark.table(GROUP_STATUS_TABLE).where("is_done = 1").count()

print(f"\n[GROUP STATUS] 전체={total_groups_all}, 완료={done_groups}, 대기={len(target_groups)}")

if spark.catalog.tableExists(DETAIL_TABLE):
    detail_cnt = spark.table(DETAIL_TABLE).count()
    print(f"[DETAIL_TABLE] 기존 분류 {detail_cnt:,}건")

if spark.catalog.tableExists(V1_DETAIL_TABLE):
    v1_cnt = spark.table(V1_DETAIL_TABLE).where("_row_id IS NOT NULL").count()
    print(f"[V1_DETAIL]    v1 결과 {v1_cnt:,}건 (_row_id IS NOT NULL)")

In [0]:
# ============================================================
# 3-2) 그룹별 샘플링 커버리지 현황
#      - 전체 메모 vs 기분류 메모 vs 남은 메모
#      - 전수 맵핑 완료(100%) 그룹 표시
#      - 제외: cate_1_depth에 '***' 포함, sc_measurement = 0
# ============================================================

# LLM 분류 대상 필터 조건
GROUP_FILTER = "cate_1_depth NOT LIKE '%***%' AND sc_measurement != 0"

# 1. 소스 테이블 그룹별 전체 메모 수 (대상 그룹만)
source_counts = spark.sql(f"""
    SELECT cate_1_depth, cate_2_depth, sc_measurement,
           COUNT(DISTINCT memo) as total_memos
    FROM {SOURCE_TABLE}
    WHERE memo IS NOT NULL AND LENGTH(TRIM(memo)) > 0
      AND {GROUP_FILTER}
    GROUP BY cate_1_depth, cate_2_depth, sc_measurement
""")

# 2. 기분류 메모 수 (v1 + 이전 회차 + 현재 회차 UNION)
classified_unions = []
all_tables = PREV_DETAIL_TABLES + [V1_DETAIL_TABLE]
if spark.catalog.tableExists(DETAIL_TABLE):
    all_tables.append(DETAIL_TABLE)

for tbl in all_tables:
    if spark.catalog.tableExists(tbl):
        classified_unions.append(
            f"SELECT DISTINCT memo, cate_1_depth, cate_2_depth, sc_measurement FROM {tbl} WHERE {GROUP_FILTER}"
        )
        print(f"  [포함] {tbl}")

if classified_unions:
    union_sql = " UNION ".join(classified_unions)
    classified_counts = spark.sql(f"""
        SELECT cate_1_depth, cate_2_depth, sc_measurement,
               COUNT(DISTINCT memo) as classified_memos
        FROM ({union_sql})
        GROUP BY cate_1_depth, cate_2_depth, sc_measurement
    """)
else:
    classified_counts = (
        source_counts
        .select("cate_1_depth", "cate_2_depth", "sc_measurement")
        .withColumn("classified_memos", F.lit(0))
    )

# 3. 조인 + 커버리지 계산
status_df = (
    source_counts
    .join(classified_counts, on=["cate_1_depth", "cate_2_depth", "sc_measurement"], how="left")
    .fillna(0, subset=["classified_memos"])
    .withColumn("remaining", F.col("total_memos") - F.col("classified_memos"))
    .withColumn("coverage_pct", F.round(F.col("classified_memos") / F.col("total_memos") * 100, 2))
    .withColumn("status",
        F.when(F.col("remaining") == 0, "✅ 전수완료")
         .when(F.col("coverage_pct") >= 50, "🟡 50%+")
         .otherwise("⏳ 진행중")
    )
    .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement")
)

status_df.cache()

# 4. 요약 통계
total_groups = status_df.count()
fully_done = status_df.where("remaining = 0").count()
partially_done = status_df.where("remaining > 0 AND classified_memos > 0").count()
not_started = status_df.where("classified_memos = 0").count()

total_memos_all = status_df.agg(F.sum("total_memos")).collect()[0][0] or 0
total_classified_all = status_df.agg(F.sum("classified_memos")).collect()[0][0] or 0
overall_pct = round(total_classified_all / total_memos_all * 100, 2) if total_memos_all > 0 else 0

print(f"\n{'='*70}")
print(f"[샘플링 커버리지 현황] Round {SAMPLING_ROUND} 실행 전")
print(f"  제외: cate_1_depth에 '***' 포함, sc_measurement = 0")
print(f"{'='*70}")
print(f"  LLM 대상 그룹: {total_groups}")
print(f"  ✅ 전수 완료: {fully_done}개")
print(f"  🟡 부분 샘플링: {partially_done}개")
print(f"  ⏳ 미시작: {not_started}개")
print(f"  전체 메모: {total_memos_all:,}건")
print(f"  분류 완료: {total_classified_all:,}건 ({overall_pct}%)")
print(f"  남은 메모: {total_memos_all - total_classified_all:,}건")
print(f"{'='*70}")

# 5. 전수 완료 그룹 출력
if fully_done > 0:
    print(f"\n[✅ 전수 맵핑 완료 그룹 ({fully_done}개)] - 파이프라인에서 자동 SKIP 됨")
    display(
        status_df.where("remaining = 0")
        .select("cate_1_depth", "cate_2_depth", "sc_measurement",
                "total_memos", "classified_memos", "coverage_pct", "status")
        .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement")
    )
else:
    print(f"\n[전수 완료 그룹 없음]")

# 6. LLM 대상 그룹 상세 테이블
print(f"\n[LLM 대상 그룹 상세]")
display(
    status_df
    .select("cate_1_depth", "cate_2_depth", "sc_measurement",
            "total_memos", "classified_memos", "remaining", "coverage_pct", "status")
)

status_df.unpersist()

In [0]:
# ============================================================
# 4) Category Feature Patterns (v5 동일)
# ============================================================

COMMON_FEATURE_PATTERNS = [
    "quality", "picture", "image", "visual", "sound", "audio", "video", "screen", "display", "tv",
    "setup", "install", "installation", "manual", "guide", "weight", "stand", "mount", "wall",
    "cable", "port", "hdmi", "bluetooth", "wifi", "wireless", "connect", "connection", "compatibility",
    "os", "software", "menu", "ui", "app", "apps", "channel", "content", "voice", "game", "gaming",
    "iot", "mobile", "brand", "service", "support", "warranty", "energy", "efficiency", "price",
    "value", "design", "color", "material", "finish", "heat", "durability", "panel", "glare",
    "angle", "brightness", "contrast", "resolution", "sharp", "clarity", "bass", "surround",
    "화질", "음질", "사운드", "화면", "디스플레이", "설치", "세팅", "매뉴얼", "가이드", "무게",
    "스탠드", "벽걸이", "선정리", "단자", "블루투스", "와이파이", "연결", "호환", "소프트웨어",
    "메뉴", "ui", "앱", "채널", "콘텐츠", "음성", "게임", "iot", "모바일", "브랜드", "서비스",
    "보증", "에너지", "효율", "가격", "가성비", "디자인", "색상", "소재", "마감", "발열",
    "내구성", "패널", "반사", "시야각", "밝기", "명암", "해상도", "선명", "저음", "서라운드"
]

CATEGORY_FEATURE_PATTERNS = {
    ("01. 사이즈", "01-01. TV 사이즈"): [
        "size", "inch", "inches", "big", "large", "small", "fit", "fits",
        "사이즈", "크기", "인치", "대형", "소형", "공간", "잘맞"
    ],
    ("02. 화질", "02-01. 선명도"): [
        "sharp", "sharpness", "clear", "clarity", "crisp", "blur", "blurry",
        "선명", "선명도", "또렷", "흐림", "블러"
    ],
    ("02. 화질", "02-02. 컴러"): [
        "color", "colour", "vivid", "vibrant", "lifelike", "saturation",
        "컴러", "색감", "생생", "채도"
    ],
    ("02. 화질", "02-03. 밝기"): [
        "bright", "brightness", "dim", "luminous",
        "밝기", "밝음", "어두움"
    ],
    ("02. 화질", "02-04. 명암비"): [
        "contrast", "black level", "black", "white balance",
        "명암", "명암비", "블랙", "검정"
    ],
    ("02. 화질", "02-05. 해상도"): [
        "resolution", "4k", "uhd", "hdr", "detail", "pixel",
        "해상도", "4k", "uhd", "hdr", "디테일", "픽셀"
    ],
    ("02. 화질", "02-06. 움직이는 영상 표현"): [
        "motion", "smooth", "blur", "judder", "stutter", "fast scene",
        "움직임", "모션", "부드러움", "잔상", "버벡"
    ],
    ("02. 화질", "02-08. 시야각"): [
        "angle", "viewing angle", "off angle",
        "시야각", "각도"
    ],
    ("02. 화질", "02-09. 반사율"): [
        "glare", "reflection", "reflective", "anti glare",
        "반사", "반사율", "빛반사", "눈부심"
    ],
    ("02. 화질", "02-10. 화질세팅"): [
        "setting", "mode", "calibration", "preset",
        "설정", "세팅", "모드", "보정"
    ],
    ("02. 화질", "02-10. 화질세팅_(1)화질 모드"): [
        "picture mode", "mode", "preset",
        "화질모드", "모드", "프리셋"
    ],
    ("02. 화질", "02-20. 전반적 화질"): [
        "picture", "image", "visual", "screen quality",
        "화질", "화면", "영상"
    ],
    ("03. 음질", "03-01. 출력"): [
        "volume", "loud", "output", "speaker power",
        "출력", "볼륨", "소리크기"
    ],
    ("03. 음질", "03-02. 선명한 음질"): [
        "clear sound", "clarity", "crisp audio",
        "선명", "맑음", "또렷"
    ],
    ("03. 음질", "03-03. 음질 세팅"): [
        "sound mode", "setting", "equalizer", "eq",
        "설정", "세팅", "모드", "eq"
    ],
    ("03. 음질", "03-04. 서라운드"): [
        "surround", "immersive", "spatial",
        "서라운드", "공간감"
    ],
    ("03. 음질", "03-05. 저음/베이스"): [
        "bass", "low end", "deep sound",
        "저음", "베이스"
    ],
    ("03. 음질", "03-20. 전반적 음질"): [
        "sound", "audio", "speaker",
        "소리", "음질", "사운드"
    ],
    ("04. 디자인", "04-01. 옆면 두께"): [
        "thin", "thickness", "slim",
        "얇음", "두께", "슬림"
    ],
    ("04. 디자인", "04-02. 베젤(프레임) 두께"): [
        "bezel", "frame",
        "베젤", "프레임"
    ],
    ("04. 디자인", "04-03. 스탠드 높이/형태"): [
        "stand", "height", "base",
        "스탠드", "높이", "받침"
    ],
    ("04. 디자인", "04-04. 벽걸이 디자인"): [
        "wall mount", "mount", "flush",
        "벽걸이", "마운트"
    ],
    ("04. 디자인", "04-05. 소재"): [
        "material", "texture",
        "소재", "재질"
    ],
    ("04. 디자인", "04-06. 색상"): [
        "color", "colour",
        "색상", "색"
    ],
    ("04. 디자인", "04-07. 마감"): [
        "finish", "build quality",
        "마감", "완성도"
    ],
    ("04. 디자인", "04-08. 후면부 디자인"): [
        "rear", "back design", "back panel",
        "후면", "뒷면"
    ],
    ("04. 디자인", "04-20. 전반적 디자인"): [
        "beautiful", "stylish", "look", "appearance",
        "예쁨", "외관", "디자인"
    ],
    ("05. 설치/세팅", "05-01. 세팅"): [
        "setup", "set up", "configure",
        "설치", "세팅", "설정"
    ],
    ("05. 설치/세팅", "05-02. 매뉴얼/가이드"): [
        "manual", "guide", "instruction",
        "매뉴얼", "가이드", "설명서"
    ],
    ("05. 설치/세팅", "05-03. 무게"): [
        "weight", "heavy", "light",
        "무게", "무겁", "가벼"
    ],
    ("05. 설치/세팅", "05-04. 선처리"): [
        "cable", "wire", "cable management",
        "선정리", "케이블", "선"
    ],
    ("05. 설치/세팅", "05-05. 각도 조절(벽걸이)"): [
        "angle", "tilt", "swivel",
        "각도", "틸트", "회전"
    ],
    ("05. 설치/세팅", "05-06. 벽걸이 설치용이성"): [
        "wall mount", "mounting",
        "벽걸이", "설치"
    ],
    ("05. 설치/세팅", "05-07. 스탠드 설치용이성"): [
        "stand assembly", "stand install",
        "스탠드", "조립"
    ],
    ("05. 설치/세팅", "05-20. 전반적 설치용이성"): [
        "install", "installation", "easy setup",
        "설치", "세팅", "조립"
    ],
    ("06. 연결성", "06-01. 연결기기 호환성"): [
        "compatible", "compatibility", "device", "devices",
        "호환", "호환성", "기기"
    ],
    ("06. 연결성", "06-02. 무선 연결성"): [
        "wifi", "wireless", "bluetooth", "pairing",
        "와이파이", "무선", "블루투스", "페어링"
    ],
    ("06. 연결성", "06-03. 연결단자 지원/개수"): [
        "hdmi", "usb", "port", "ports",
        "단자", "포트", "hdmi", "usb"
    ],
    ("06. 연결성", "06-04. (편리한)단자 위치"): [
        "port location", "easy access",
        "단자위치", "접근"
    ],
    ("06. 연결성", "06-20. 전반적 연결성"): [
        "connect", "connection",
        "연결", "연결성"
    ],
    ("07. 스마트 사용성", "07-01. 채널/컨텐츠 APP"): [
        "app", "apps", "channel", "content", "streaming", "ott",
        "앱", "채널", "콘텐츠", "스트리밍", "ott"
    ],
    ("07. 스마트 사용성", "07-02. 구동성/구동속도"): [
        "fast", "slow", "lag", "speed", "loading",
        "속도", "빠름", "느림", "로딩", "렉"
    ],
    ("07. 스마트 사용성", "07-02. 구동성/구동속도_(1)TV 전반"): [
        "fast", "slow", "lag", "speed", "tv response",
        "속도", "빠름", "느림", "반응", "렉"
    ],
    ("07. 스마트 사용성", "07-02. 구동성/구동속도_(2)APP/웹전반"): [
        "app speed", "web speed", "lag", "loading",
        "앱속도", "웹속도", "로딩", "렉"
    ],
    ("07. 스마트 사용성", "07-03. 메뉴 구성/UI"): [
        "menu", "ui", "interface", "navigation",
        "메뉴", "ui", "인터페이스", "탐색"
    ],
    ("07. 스마트 사용성", "07-04. SW/OS"): [
        "os", "software", "update", "bug",
        "os", "소프트웨어", "업데이트", "버그"
    ],
    ("07. 스마트 사용성", "07-05. 컨텐츠 탐색 사용성"): [
        "search", "browse", "discover",
        "탐색", "검색", "브라우징"
    ],
    ("07. 스마트 사용성", "07-06. 리모컨 사용성"): [
        "remote", "control", "button", "layout", "pointer", "backlight", "easy",
        "리모컨", "조작", "버튼", "레이아웃", "포인터", "백라이트", "편리"
    ],
    ("07. 스마트 사용성", "07-07. 리모컨 디자인"): [
        "remote design", "remote look",
        "리모컨 디자인", "외관"
    ],
    ("07. 스마트 사용성", "07-08. 음성 인식/조작"): [
        "voice", "speech", "assistant",
        "음성", "음성인식", "음성조작"
    ],
    ("07. 스마트 사용성", "07-09. 게임 기능"): [
        "game", "gaming", "latency",
        "게임", "게이밍", "지연"
    ],
    ("07. 스마트 사용성", "07-10. 부가 기능"): [
        "feature", "features", "extra function",
        "기능", "부가기능"
    ],
    ("07. 스마트 사용성", "07-11. 홈 IoT"): [
        "iot", "smart home",
        "iot", "스마트홈"
    ],
    ("07. 스마트 사용성", "07-12. 모바일 연동"): [
        "mobile", "phone", "cast", "mirror",
        "모바일", "휴대폰", "캐스트", "미러링"
    ],
    ("07. 스마트 사용성", "07-13. 광고"): [
        "ad", "ads", "advertisement",
        "광고"
    ],
    ("07. 스마트 사용성", "07-20. 전반적 스마트 사용성"): [
        "smart", "usability", "easy to use",
        "스마트", "사용성", "편리"
    ],
    ("08. 내구성", "08-01. A/S"): [
        "service", "support", "repair", "warranty",
        "as", "서비스", "지원", "수리", "보증"
    ],
    ("08. 내구성", "08-02. 품질보증기간"): [
        "warranty", "guarantee",
        "보증", "보증기간"
    ],
    ("08. 내구성", "08-03. 잔상"): [
        "ghosting", "burn in", "image retention",
        "잔상", "번인"
    ],
    ("08. 내구성", "08-04. 패널 내구성"): [
        "panel", "durability", "screen life",
        "패널", "내구성"
    ],
    ("08. 내구성", "08-05. 발열"): [
        "heat", "heating", "hot",
        "발열", "뜨거움"
    ],
    ("08. 내구성", "08-20. 전반적 내구성"): [
        "durable", "reliable", "lasting",
        "내구성", "튼튼", "오래감"
    ],
    ("09. 친환경", "09-01. 에너지 효율"): [
        "energy", "efficient", "efficiency", "power saving",
        "에너지", "효율", "절전"
    ],
    ("09. 친환경", "09-02. 친환경 소재"): [
        "eco", "eco-friendly", "material",
        "친환경", "소재"
    ],
    ("10. 가격", "10-01. 가격/가격대비 가치"): [
        "price", "value", "worth", "deal", "budget", "affordable",
        "가격", "가성비", "값어치", "딜", "저렴"
    ],
    ("11. 브랜드", "11-01. 브랜드"): [
        "brand", "samsung", "lg", "sony", "tcl", "hisense",
        "브랜드", "삼성", "엘지", "소니"
    ],
    ("12. 품질 불량", "12-01. 화질 불량"): [
        "defect", "screen issue", "dead pixel", "dark", "blurry",
        "불량", "화질불량", "데드픽셀", "암부", "흐림"
    ],
    ("12. 품질 불량", "12-02. 제품 불량"): [
        "defective", "broken", "fault", "problem",
        "불량", "고장", "문제"
    ],
    ("12. 품질 불량", "12-03. 설치 불량"): [
        "installation issue", "bad install",
        "설치불량", "설치문제"
    ],
    ("12. 품질 불량", "12-04. 기능 불량"): [
        "malfunction", "doesn't work", "failed", "not working",
        "기능불량", "작동안함", "먹통", "고장"
    ],
    ("12. 품질 불량", "12-05. 기타 불량"): [
        "issue", "problem", "fault",
        "불량", "문제", "이슈"
    ],
    ("13. 전반적 평가", "13-01. 전반적 평가"): [
        "overall", "generally", "satisfied", "happy", "purchase", "product",
        "전반적", "전체적", "만족", "구매", "제품"
    ],
}


def get_feature_patterns(cate_1_depth: str, cate_2_depth: str) -> List[str]:
    return COMMON_FEATURE_PATTERNS + CATEGORY_FEATURE_PATTERNS.get((cate_1_depth, cate_2_depth), [])


def has_specific_reason_for_category(text: str, clue_keywords: List[str], cate_1_depth: str, cate_2_depth: str) -> bool:
    memo = clean_text(text).lower()
    if any(clean_text(k).lower() in memo for k in clue_keywords if clean_text(k)):
        return True
    feature_patterns = get_feature_patterns(cate_1_depth, cate_2_depth)
    return any(clean_text(p).lower() in memo for p in feature_patterns if clean_text(p))


def should_be_overall_for_category(text: str, clue_keywords: List[str], generic_terms: List[str], cate_1_depth: str, cate_2_depth: str) -> bool:
    memo = clean_text(text).lower()
    if not memo:
        return False
    if len(memo) > 14:
        return False
    if has_specific_reason_for_category(memo, clue_keywords, cate_1_depth, cate_2_depth):
        return False
    if len(re.findall(r"[A-Za-z\uac00-\ud7a3]+", memo)) > 3:
        return False
    return any(clean_text(t).lower() in memo for t in generic_terms if clean_text(t))


# ============================================================
# Overall Sentiment Detection  ("전반적" 그룹 전용 rule pre-filter)
# ============================================================
# 전반적 긍정/부정 = length<25 + 감정 표현만 + 피처 설명 단어 없음

# 감정 전용 표현 (형용사 / 부사)
SENTIMENT_EXPRESSIONS = {
    # --- Korean ---
    "좋아요", "좋아", "좋음", "좋다", "좋습니다", "좋네요", "좋았", "좋은",
    "최고", "만족", "대만족", "훌륭", "완벽", "추천", "강추", "굿",
    "괜찮", "맘에", "마음에", "만족스러", "괴다",
    "별로", "나쁨", "나빠", "나쁜", "불만", "실망", "후회", "최악",
    "싫어", "아쉬", "그저그래", "그냥", "보통",
    "너무", "정말", "아주", "매우", "진짜",
    # --- English ---
    "good", "great", "excellent", "awesome", "amazing", "perfect",
    "wonderful", "fantastic", "love", "loved", "best", "nice",
    "superb", "outstanding", "incredible", "brilliant",
    "bad", "terrible", "awful", "horrible", "worst", "poor",
    "suck", "sucks", "disappointed", "disappointing", "hate", "ugly",
    "useless", "okay", "ok", "fine", "not bad", "rubbish", "crap",
    "very", "really", "so", "super", "quite", "absolutely", "totally",
    "highly", "extremely", "pretty", "just",
    # --- German ---
    "gut", "toll", "super", "perfekt", "schlecht", "ausgezeichnet", "sehr",
    # --- Spanish ---
    "bueno", "malo", "excelente", "genial", "perfecto", "muy",
}

# Stopwords / filler (의미 없는 단어 - 무시 가능)
FILLER_WORDS = {
    "it", "is", "was", "the", "a", "an", "i", "my", "this", "that",
    "its", "im", "am", "are", "be", "been", "being",
    "not", "no", "yes", "yeah", "yep", "nope",
    "에요", "이에요", "입니다", "해요", "요", "네요",
    "다", "은", "는", "이", "가", "를", "을", "도", "만",
    "데", "에", "서", "로",
}

# 피처/설명적 단어 (이 단어가 있으면 전반적 긍정/부정이 아님)
FEATURE_EXCLUSION_WORDS = [
    # Size
    "크", "작", "커", "작아", "big", "small", "large", "tiny", "huge", "사이즈", "인치",
    # Brightness / Darkness
    "밝", "어두", "밝아", "어두워", "bright", "dark", "dim",
    # Sound volume
    "시끄러", "조용", "loud", "quiet", "silent",
    # Speed
    "빠르", "느리", "fast", "slow",
    # Weight
    "무겁", "가벼", "heavy", "light",
    # Sharpness
    "선명", "흐릿", "sharp", "clear", "blurry",
    # Thickness
    "얇", "두꺼", "thin", "thick",
    # Color
    "색감", "색상", "color", "colour",
    # Feature categories
    "화질", "음질", "화면", "소리", "사운드", "디자인", "설치",
    "리모컨", "앱", "게임", "연결", "wifi", "블루투스", "hdmi",
    "패널", "베젤", "스탠드", "가격", "배송", "보증",
    "picture", "sound", "audio", "screen", "display", "remote",
    "price", "delivery", "install", "panel", "bezel",
]


def is_pure_overall_sentiment(memo: str) -> bool:
    """
    Rule-based: 전반적 긍정/부정 판단.
    - len < 25
    - 감정 표현만으로 구성 (종결어알 · 조동사 제외, 감정 형용사/부사만)
    - 피처 설명 단어(크다/작다/밝다/어둡다 등) 포함 시 False
    """
    if not memo:
        return False
    text = memo.strip()
    if len(text) >= 25:
        return False

    text_lower = text.lower()

    # 피처/설명 단어가 하나라도 있으면 → NOT overall
    for w in FEATURE_EXCLUSION_WORDS:
        if w.lower() in text_lower:
            return False

    # 토큰화 후 감정+filler 만으로 구성되는지 확인
    tokens = re.findall(r"[a-z\uac00-\ud7a3]+", text_lower)
    if not tokens:
        return False

    has_sentiment = False
    for tok in tokens:
        if tok in SENTIMENT_EXPRESSIONS:
            has_sentiment = True
        elif tok in FILLER_WORDS:
            continue
        else:
            # 감정도 filler도 아닌 단어 → 구체적 내용 있음
            return False

    return has_sentiment


print("[OK] Category feature patterns + overall sentiment rule loaded")

In [0]:
# ============================================================
# 5) Classify + Checkpoint (Round 4: LLM 결과만 저장)
#    - GROUP_STATUS_TABLE is_done=0 그룹 대상
#    - LLM 분류 → DETAIL_TABLE(회차별) 저장
#    - v1 결합은 파이프라인 완료 후 별도 머지 단계에서 수행
#    - GROUP_STATUS_TABLE is_done=1 업데이트
#    - (v4) memo_summary 컬럼 포함 저장
# ============================================================

# --------------------------------------------------
# 0. rule_map / topic_pool_group_map 로드
# --------------------------------------------------
rule_map = {
    (r["cate_1_depth"], r["cate_2_depth"], int(r["sc_measurement"])): r.asDict(recursive=True)
    for r in spark.table(RULE_PROFILE_TABLE).toLocalIterator()
}

topic_pool_group_map: Dict[tuple, List[Dict]] = {}
for row in spark.table(TOPIC_POOL_TABLE).toLocalIterator():
    key = (row["cate_1_depth"], row["cate_2_depth"], int(row["sc_measurement"]))
    reps = []
    try:
        reps = json.loads(row["representative_memos_json"]) if row["representative_memos_json"] else []
    except Exception:
        pass
    topic_pool_group_map.setdefault(key, []).append({
        "topic": row["topic"],
        "description": row["description"],
        "representative_memos": reps,
    })


# --------------------------------------------------
# 1. build_classify_messages
# --------------------------------------------------
def build_classify_messages(
    batch_rows: List[Dict[str, Any]],
    topic_pool_payload: List[Dict[str, Any]],
    rule_row: Dict[str, Any],
    cate_1_depth: str,
    cate_2_depth: str,
    sc_measurement: int,
) -> List[Dict[str, str]]:
    clue_keywords = json.loads(rule_row["clue_keywords_json"]) if rule_row["clue_keywords_json"] else []
    non_overall_examples = json.loads(rule_row["non_overall_examples_json"]) if rule_row["non_overall_examples_json"] else []
    category_patterns = get_feature_patterns(cate_1_depth, cate_2_depth)

    topic_info = []
    for t in topic_pool_payload:
        info = {"topic": t["topic"], "description": t["description"]}
        rep_memos = t.get("representative_memos", [])
        if rep_memos:
            info["examples"] = rep_memos[:3]
        topic_info.append(info)

    system = f"""
You are a VOC classifier for TV review topic classification.

Classify each memo into exactly one topic from the fixed topic list.
Every memo must belong to exactly one topic.

Rules:
- The task is to identify WHY the writer evaluated {cate_2_depth} as {sc_label(sc_measurement)}.
- Overall topic is '{clean_text(rule_row["overall_topic_label"])}'.
- {clean_text(rule_row["overall_usage_rule"])}
- {clean_text(rule_row["specific_reason_rule"])}
- clue keywords:
  {json.dumps(clue_keywords, ensure_ascii=False)}
- category fallback feature patterns:
  {json.dumps(category_patterns[:60], ensure_ascii=False)}
- non-overall examples:
  {json.dumps(non_overall_examples, ensure_ascii=False)}
- Do not invent any new topic.
- explanation must be a short Korean sentence.

Return only JSON:
{{
  "results": [
    {{
      "row_id": "",
      "topic": "",
      "explanation": ""
    }}
  ]
}}
"""
    user = (
        "Fixed topics (with description and example memos):\n"
        + json.dumps(topic_info, ensure_ascii=False)
        + "\n\nMemos:\n"
        + json.dumps(
            [{"row_id": str(r["_row_id"]), "memo": clean_text(r["memo"])} for r in batch_rows],
            ensure_ascii=False,
        )
    )
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


# --------------------------------------------------
# 2. build_batch_reclassify_messages
# --------------------------------------------------
def build_batch_reclassify_messages(
    batch_rows: List[Dict[str, Any]],
    topic_pool_payload: List[Dict[str, Any]],
    rule_row: Dict[str, Any],
    cate_1_depth: str,
    cate_2_depth: str,
) -> List[Dict[str, str]]:
    overall_topic = clean_text(rule_row["overall_topic_label"])
    specific_topic_payload = [
        {"topic": t["topic"], "description": t["description"]}
        for t in topic_pool_payload if clean_text(t["topic"]) != overall_topic
    ]
    clue_keywords = json.loads(rule_row["clue_keywords_json"]) if rule_row["clue_keywords_json"] else []
    category_patterns = get_feature_patterns(cate_1_depth, cate_2_depth)

    system = f"""
You are a VOC classifier for TV review topic classification.

These memos were incorrectly over-generalized as '{overall_topic}'.
Reclassify each memo using only the non-general topics below.

Rules:
- Choose exactly one non-general topic for each memo.
- Each memo contains a specific reason and must not remain as '{overall_topic}'.
- clue keywords:
  {json.dumps(clue_keywords, ensure_ascii=False)}
- category fallback feature patterns:
  {json.dumps(category_patterns[:60], ensure_ascii=False)}
- explanation must be a short Korean sentence.

Return only JSON:
{{
  "results": [
    {{
      "row_id": "",
      "topic": "",
      "explanation": ""
    }}
  ]
}}
"""
    user = (
        "Allowed non-general topics:\n"
        + json.dumps(specific_topic_payload, ensure_ascii=False)
        + "\n\nMemos:\n"
        + json.dumps(
            [{"row_id": str(r["_row_id"]), "memo": clean_text(r["memo"])} for r in batch_rows],
            ensure_ascii=False,
        )
    )
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


# --------------------------------------------------
# 3. classify_batch_and_merge (3-Phase)
#    NOTE: 이 함수는 Cell 5-1에서 v4 버전으로 재정의됨
# --------------------------------------------------
def classify_batch_and_merge(
    source_rows: List[Dict[str, Any]],
    topic_payload: List[Dict[str, Any]],
    rule_row: Dict[str, Any],
    c1: str, c2: str, sc: int,
    idx: int, total_groups: int,
) -> List[Dict[str, Any]]:
    clue_keywords = json.loads(rule_row["clue_keywords_json"]) if rule_row["clue_keywords_json"] else []
    generic_terms = json.loads(rule_row["generic_terms_json"]) if rule_row["generic_terms_json"] else []
    overall_topic = clean_text(rule_row["overall_topic_label"])
    total_batches = (len(source_rows) + CLASSIFY_BATCH_SIZE - 1) // CLASSIFY_BATCH_SIZE

    # Phase 1: 초기 배치 분류
    classified_rows = []
    for batch_no, batch in enumerate(chunk_list(source_rows, CLASSIFY_BATCH_SIZE), start=1):
        batch_start_ts = time.time()
        print(f"    [BATCH] {batch_no}/{total_batches}, rows={len(batch)}")
        batch_result = call_llm(
            build_classify_messages(batch, topic_payload, rule_row, c1, c2, sc)
        )
        result_map = {
            str(item.get("row_id")): item
            for item in batch_result.get("results", [])
            if isinstance(item, dict)
        }
        for row in batch:
            mapped = result_map.get(str(row["_row_id"]), {})
            classified_rows.append({
                "_row_id": row["_row_id"],
                "cate_1_depth": c1,
                "cate_2_depth": c2,
                "sc_measurement": sc,
                "memo": row["memo"],
                "topic_raw": clean_text(mapped.get("topic")),
                "explanation_raw": clean_text(mapped.get("explanation")),
            })
        print(f"    [BATCH DONE] {batch_no}/{total_batches}, {round(time.time() - batch_start_ts, 2)}s")
        time.sleep(RATE_LIMIT_SECONDS)

    # Phase 2: 배치 Overall 재분류
    reclass_candidates = [
        cr for cr in classified_rows
        if cr["topic_raw"] == overall_topic
        and has_specific_reason_for_category(cr["memo"], clue_keywords, c1, c2)
        and not should_be_overall_for_category(cr["memo"], clue_keywords, generic_terms, c1, c2)
    ]
    if reclass_candidates:
        reclass_total = len(reclass_candidates)
        reclass_batches = (reclass_total + CLASSIFY_BATCH_SIZE - 1) // CLASSIFY_BATCH_SIZE
        print(f"    [RECLASS] {reclass_total} rows -> {reclass_batches} batch(es)")
        reclass_result_map = {}
        for rb_no, rb in enumerate(chunk_list(reclass_candidates, CLASSIFY_BATCH_SIZE), start=1):
            reclass_result = call_llm(
                build_batch_reclassify_messages(rb, topic_payload, rule_row, c1, c2)
            )
            for item in reclass_result.get("results", []):
                if isinstance(item, dict) and item.get("row_id"):
                    reclass_result_map[str(item["row_id"])] = item
            time.sleep(RATE_LIMIT_SECONDS)
        applied = 0
        for cr in classified_rows:
            mapped = reclass_result_map.get(str(cr["_row_id"]))
            if mapped:
                retry_topic = clean_text(mapped.get("topic"))
                if retry_topic and retry_topic != overall_topic:
                    cr["topic_raw"] = retry_topic
                    cr["explanation_raw"] = clean_text(mapped.get("explanation")) or cr["explanation_raw"]
                    applied += 1
        print(f"    [RECLASS DONE] {applied}/{reclass_total} reclassified")

    # Phase 3: 희소 주제 병합
    from collections import Counter
    topic_counter = Counter(cr["topic_raw"] for cr in classified_rows)
    rare_topics_sorted = sorted(
        [t for t, cnt in topic_counter.items() if cnt < MIN_ROWS_PER_TOPIC and clean_text(t)]
    )
    rare_topic_label = f"기타({', '.join(rare_topics_sorted[:3])})" if rare_topics_sorted else "기타"
    topic_desc_map = {t["topic"]: t["description"] for t in topic_payload}

    final_rows = []
    for rd in classified_rows:
        raw_topic = clean_text(rd["topic_raw"])
        raw_expl  = clean_text(rd["explanation_raw"])
        if raw_topic in rare_topics_sorted:
            final_topic = rare_topic_label
            final_explanation = f"희소 주제 묶음: {raw_topic}" if raw_topic else "희소 주제 묶음"
        else:
            final_topic = raw_topic if raw_topic else "기타"
            final_explanation = raw_expl or topic_desc_map.get(final_topic, "")
        final_rows.append({
            "_row_id": rd["_row_id"],
            "cate_1_depth": c1, "cate_2_depth": c2, "sc_measurement": sc,
            "memo": rd["memo"], "topic": final_topic, "description": final_explanation,
        })
    return final_rows


# --------------------------------------------------
# 4. Main classify loop (Round 4: LLM 결과만 저장)
#    - v1 결합 없음 (회차별 분리 저장, 마지막에 머지)
#    - delete_group_rows 없음 (새 테이블이므로 불필요)
#    - (v4) memo_summary 컬럼 포함 저장
# --------------------------------------------------
pipeline_start_ts = time.time()
total_pending = len(target_groups)
processed = 0
skipped = 0

print(f"{'='*70}")
print(f"[PIPELINE START] Round {SAMPLING_ROUND} | {total_pending} pending groups")
print(f"  DETAIL_TABLE = {DETAIL_TABLE}")
print(f"  MEMO_SUMMARIZE_THRESHOLD = {MEMO_SUMMARIZE_THRESHOLD}자")
print(f"{'='*70}")

for idx, g in enumerate(target_groups, start=1):
    c1 = g["cate_1_depth"]
    c2 = g["cate_2_depth"]
    sc = int(g["sc_measurement"])
    key = (c1, c2, sc)

    # rule/topic pool 없으면 스킵
    if key not in rule_map or key not in topic_pool_group_map:
        skipped += 1
        print(f"[SKIP] {idx}/{total_pending} | no rule/topic pool | {c2}")
        continue

    group_start_ts = time.time()

    # 신규 메모 건수 (이전 회차 + v1 제외)
    new_count = get_group_new_count(c1, c2, sc)

    print(f"\n[START] {idx}/{total_pending} | {c2} | sc={sc}")
    print(f"  new_memos={new_count:,} (이전 회차 + v1 제외)")

    # 신규 메모가 없으면 스킵
    if new_count == 0:
        skipped += 1
        update_group_status(c1, c2, sc, is_done=1)
        print(f"  [SKIP] 신규 메모 없음")
        continue

    try:
        rule_row = rule_map[key]
        topic_payload = topic_pool_group_map[key]
        is_overall_group = "전반적" in c2

        # ==================================================
        # A. LLM 분류 (신규 메모만)
        # ==================================================
        llm_rows = []
        if is_overall_group:
            all_memos_df = load_group_all_memos_df(c1, c2, sc)
            total_count = all_memos_df.count()

            is_overall_udf = F.udf(is_pure_overall_sentiment, T.BooleanType())
            tagged_df = all_memos_df.withColumn("_is_overall", is_overall_udf(F.col("memo")))

            overall_df = tagged_df.where(F.col("_is_overall") == True).select("memo")
            non_overall_df = tagged_df.where(F.col("_is_overall") == False).select("memo")

            overall_count = overall_df.count()
            non_overall_count = non_overall_df.count()
            print(f"  [전반적 PRE-FILTER] total={total_count}, overall={overall_count}, non_overall={non_overall_count}")

            overall_topic_name = clean_text(rule_row["overall_topic_label"])
            for r in overall_df.toLocalIterator():
                llm_rows.append({
                    "_row_id": -1, "cate_1_depth": c1, "cate_2_depth": c2,
                    "sc_measurement": sc, "memo": r["memo"],
                    "memo_summary": "",  # v4: 규칙 기반 자동분류는 요약 불필요
                    "topic": overall_topic_name,
                    "description": "순수 감정 표현 (rule 기반 자동 분류)",
                })

            if non_overall_count > 0:
                sample_size = get_dynamic_sample_size(non_overall_count)
                print(f"  [샘플] {non_overall_count}건 -> {sample_size}건")
                sampled_df = (
                    non_overall_df
                    .withColumn("_rn", F.row_number().over(
                        Window.orderBy(F.md5(F.concat(
                            F.coalesce(F.col("memo"), F.lit("")),
                            F.lit(f"||{c1}||{c2}||{sc}||seed_20260420_round{SAMPLING_ROUND}")
                        )))))
                    .where(F.col("_rn") <= sample_size).drop("_rn")
                    .withColumn("_row_id", F.monotonically_increasing_id())
                ).cache()
                source_rows = [r.asDict(recursive=True) for r in sampled_df.toLocalIterator()]
                if source_rows:
                    llm_rows += classify_batch_and_merge(
                        source_rows, topic_payload, rule_row, c1, c2, sc, idx, total_pending
                    )
                sampled_df.unpersist()
        else:
            sample_size = get_dynamic_sample_size(new_count)
            print(f"  [샘플] {new_count:,}건 -> {sample_size}건")
            sample_df = load_group_sample_df(c1, c2, sc, sample_size)
            source_rows = [r.asDict(recursive=True) for r in sample_df.toLocalIterator()]
            if source_rows:
                llm_rows = classify_batch_and_merge(
                    source_rows, topic_payload, rule_row, c1, c2, sc, idx, total_pending
                )
            sample_df.unpersist()

        # ==================================================
        # B. DETAIL_TABLE 저장 (LLM 결과만, append)
        #    - 새 회차 테이블이므로 delete 불필요
        #    - v1 결합 없음 (마지막에 머지)
        #    - (v4) memo_summary 컬럼 보장
        # ==================================================
        if llm_rows:
            # v4: memo_summary 컬럼 보장 (모든 행에 존재하도록)
            for row in llm_rows:
                row.setdefault("memo_summary", "")
            llm_df = spark.createDataFrame(pd.DataFrame(llm_rows))
            append_table(llm_df, DETAIL_TABLE)

        # ==================================================
        # C. GROUP_STATUS_TABLE is_done=1
        # ==================================================
        update_group_status(c1, c2, sc, is_done=1)

        # ==================================================
        # D. 체크포인트 출력
        # ==================================================
        processed += 1
        elapsed = round(time.time() - group_start_ts, 2)
        total_elapsed = round(time.time() - pipeline_start_ts)
        remaining = total_pending - idx
        avg_per_group = total_elapsed / processed if processed else 0
        eta = round(remaining * avg_per_group)

        log_progress(c1, c2, sc, f"classify_round{SAMPLING_ROUND}", "done",
                     f"llm={len(llm_rows)}")

        print(f"  [SAVE] LLM={len(llm_rows)} -> {DETAIL_TABLE}")
        print(f"  [DONE] {idx}/{total_pending} | {elapsed}s")
        print(f"  [CHECKPOINT] processed={processed}, skipped={skipped}, "
              f"elapsed={total_elapsed}s, ETA\u2248{eta}s")

    except Exception as exc:
        log_failure(c1, c2, sc, f"classify_round{SAMPLING_ROUND}", repr(exc))
        print(f"  [FAILED] {idx}/{total_pending} | {c2} | {repr(exc)[:200]}")

# --------------------------------------------------
# Pipeline 완료 요약
# --------------------------------------------------
total_elapsed = round(time.time() - pipeline_start_ts)
print(f"\n{'='*70}")
print(f"[PIPELINE COMPLETE] Round {SAMPLING_ROUND}")
print(f"  processed={processed}, skipped={skipped}, total_elapsed={total_elapsed}s")

if spark.catalog.tableExists(DETAIL_TABLE):
    total_detail = spark.table(DETAIL_TABLE).count()
    print(f"  {DETAIL_TABLE} total = {total_detail:,}")

done_now = spark.table(GROUP_STATUS_TABLE).where("is_done=1").count()
pending_now = spark.table(GROUP_STATUS_TABLE).where("is_done=0").count()
print(f"  GROUP_STATUS: done={done_now}, pending={pending_now}")
print(f"{'='*70}")
print(f"\n\u2192 \ub2e4\uc74c \ub2e8\uacc4: \ud68c\ucc28\ubcc4 \uacb0\uacfc\ub97c V1_DETAIL_TABLE\uc5d0 \uba38\uc9c0\ud558\ub824\uba74 Cell 7 \uc2e4\ud589")

In [0]:
# ============================================================
# 5-1) classify_batch_and_merge 재정의: v4 메모 요약 + Phase 3 기타 병합 제거
#      - 200자 초과 메모 → 1문장 요약 후 요약본으로 분류
#      - LLM이 선정한 토픽 그대로 유지 (희소 주제도 병합 안 함)
#      - topic_raw가 빈 값일 경우만 "기타"로 fallback
#      - memo_summary 컬럼 추가 (DETAIL_TABLE 저장용)
# ============================================================

def classify_batch_and_merge(
    source_rows: List[Dict[str, Any]],
    topic_payload: List[Dict[str, Any]],
    rule_row: Dict[str, Any],
    c1: str, c2: str, sc: int,
    idx: int, total_groups: int,
) -> List[Dict[str, Any]]:
    clue_keywords = json.loads(rule_row["clue_keywords_json"]) if rule_row["clue_keywords_json"] else []
    generic_terms = json.loads(rule_row["generic_terms_json"]) if rule_row["generic_terms_json"] else []
    overall_topic = clean_text(rule_row["overall_topic_label"])
    total_batches = (len(source_rows) + CLASSIFY_BATCH_SIZE - 1) // CLASSIFY_BATCH_SIZE

    # ==================================================
    # Phase 0 (v4 신규): 메모 요약
    #   - 200자 초과 메모 → LLM 1문장 요약
    #   - 요약본으로 분류 수행
    # ==================================================
    classify_rows, summary_map = prepare_rows_for_classification(source_rows)
    summarized_count = len(summary_map)
    if summarized_count > 0:
        print(f"    [v4 요약] {summarized_count}건 메모 요약 완료, 요약본으로 분류 진행")

    # Phase 1: 초기 배치 분류 (요약본 사용)
    classified_rows = []
    for batch_no, batch in enumerate(chunk_list(classify_rows, CLASSIFY_BATCH_SIZE), start=1):
        batch_start_ts = time.time()
        print(f"    [BATCH] {batch_no}/{total_batches}, rows={len(batch)}")
        batch_result = call_llm(
            build_classify_messages(batch, topic_payload, rule_row, c1, c2, sc)
        )
        result_map = {
            str(item.get("row_id")): item
            for item in batch_result.get("results", [])
            if isinstance(item, dict)
        }
        for row in batch:
            mapped = result_map.get(str(row["_row_id"]), {})
            classified_rows.append({
                "_row_id": row["_row_id"],
                "cate_1_depth": c1,
                "cate_2_depth": c2,
                "sc_measurement": sc,
                "memo": row["memo"],  # 이 시점에서는 요약본일 수 있음
                "topic_raw": clean_text(mapped.get("topic")),
                "explanation_raw": clean_text(mapped.get("explanation")),
            })
        print(f"    [BATCH DONE] {batch_no}/{total_batches}, {round(time.time() - batch_start_ts, 2)}s")
        time.sleep(RATE_LIMIT_SECONDS)

    # Phase 2: 배치 Overall 재분류
    reclass_candidates = [
        cr for cr in classified_rows
        if cr["topic_raw"] == overall_topic
        and has_specific_reason_for_category(cr["memo"], clue_keywords, c1, c2)
        and not should_be_overall_for_category(cr["memo"], clue_keywords, generic_terms, c1, c2)
    ]
    if reclass_candidates:
        reclass_total = len(reclass_candidates)
        reclass_batches = (reclass_total + CLASSIFY_BATCH_SIZE - 1) // CLASSIFY_BATCH_SIZE
        print(f"    [RECLASS] {reclass_total} rows -> {reclass_batches} batch(es)")
        reclass_result_map = {}
        for rb_no, rb in enumerate(chunk_list(reclass_candidates, CLASSIFY_BATCH_SIZE), start=1):
            reclass_result = call_llm(
                build_batch_reclassify_messages(rb, topic_payload, rule_row, c1, c2)
            )
            for item in reclass_result.get("results", []):
                if isinstance(item, dict) and item.get("row_id"):
                    reclass_result_map[str(item["row_id"])] = item
            time.sleep(RATE_LIMIT_SECONDS)
        applied = 0
        for cr in classified_rows:
            mapped = reclass_result_map.get(str(cr["_row_id"]))
            if mapped:
                retry_topic = clean_text(mapped.get("topic"))
                if retry_topic and retry_topic != overall_topic:
                    cr["topic_raw"] = retry_topic
                    cr["explanation_raw"] = clean_text(mapped.get("explanation")) or cr["explanation_raw"]
                    applied += 1
        print(f"    [RECLASS DONE] {applied}/{reclass_total} reclassified")

    # Output: LLM 선정 토픽 그대로 사용 (기타 병합 없음) + memo_summary 컬럼 추가
    topic_desc_map = {t["topic"]: t["description"] for t in topic_payload}
    
    # source_rows의 원본 memo 복원용 맵
    original_memo_map = {str(r["_row_id"]): clean_text(r["memo"]) for r in source_rows}
    
    final_rows = []
    for rd in classified_rows:
        raw_topic = clean_text(rd["topic_raw"])
        raw_expl = clean_text(rd["explanation_raw"])
        final_topic = raw_topic if raw_topic else "기타"
        final_explanation = raw_expl or topic_desc_map.get(final_topic, "")
        
        row_id_str = str(rd["_row_id"])
        original_memo = original_memo_map.get(row_id_str, rd["memo"])
        memo_summary = summary_map.get(row_id_str)  # None if not summarized
        
        final_rows.append({
            "_row_id": rd["_row_id"],
            "cate_1_depth": c1, "cate_2_depth": c2, "sc_measurement": sc,
            "memo": original_memo,  # 항상 원본 메모 저장
            "memo_summary": memo_summary or "",  # 요약본 (요약 안 된 경우 빈 문자열)
            "topic": final_topic,
            "description": final_explanation,
        })
    return final_rows

print("[OK] classify_batch_and_merge 재정의 완료 (v4: 메모 요약 + Phase 3 기타 병합 제거)")

In [0]:
# ============================================================
# 5-2) build_classify_messages Override: "기타" + "오분류" 토픽 추가
#      - 어느 주제에도 속하지 않는 메모 → "기타" (description에 사유)
#      - 감성 불일치 / 속성 불일치 → "오분류" (description에 사유)
# ============================================================

def build_classify_messages(
    batch_rows: List[Dict[str, Any]],
    topic_pool_payload: List[Dict[str, Any]],
    rule_row: Dict[str, Any],
    cate_1_depth: str,
    cate_2_depth: str,
    sc_measurement: int,
) -> List[Dict[str, str]]:
    clue_keywords = json.loads(rule_row["clue_keywords_json"]) if rule_row["clue_keywords_json"] else []
    non_overall_examples = json.loads(rule_row["non_overall_examples_json"]) if rule_row["non_overall_examples_json"] else []
    category_patterns = get_feature_patterns(cate_1_depth, cate_2_depth)

    topic_info = []
    for t in topic_pool_payload:
        info = {"topic": t["topic"], "description": t["description"]}
        rep_memos = t.get("representative_memos", [])
        if rep_memos:
            info["examples"] = rep_memos[:3]
        topic_info.append(info)

    sc_label_str = sc_label(sc_measurement)  # "긍정" or "부정"
    opposite_label = "부정" if sc_measurement == 1 else "긍정"

    system = f"""You are a VOC classifier for TV review topic classification.

Classify each memo into exactly one topic from the fixed topic list,
or into the special topics "기타" or "오분류" when applicable.

=== CLASSIFICATION RULES ===

1. NORMAL CLASSIFICATION:
   - The task is to identify WHY the writer evaluated {cate_2_depth} as {sc_label_str}.
   - Choose exactly one topic from the fixed topic list.
   - Overall topic is '{clean_text(rule_row["overall_topic_label"])}'.
   - {clean_text(rule_row["overall_usage_rule"])}
   - {clean_text(rule_row["specific_reason_rule"])}

2. SPECIAL TOPIC "기타" (Others):
   Use ONLY when the memo does NOT fit ANY topic in the fixed list.
   Cases:
   - The memo discusses something completely unrelated to any listed topic.
   In explanation, write WHY none of the topics fit (Korean, 1 sentence).

3. SPECIAL TOPIC "오분류" (Misclassification):
   Use when the memo clearly should NOT belong to this group. Cases:
   a) Sentiment mismatch: This group is {sc_label_str}(sc={sc_measurement}),
      but the memo expresses {opposite_label} sentiment.
      → explanation: "감성 불일치: {sc_label_str} 그룹이나 {opposite_label} 내용"
   b) Category mismatch: The memo is NOT about {cate_1_depth}/{cate_2_depth}
      but about a completely different product attribute.
      → explanation: "속성 불일치: [actual attribute]에 대한 리뷰"
   In explanation, clearly state the misclassification reason (Korean).

=== ADDITIONAL CONTEXT ===
- clue keywords:
  {json.dumps(clue_keywords, ensure_ascii=False)}
- category fallback feature patterns:
  {json.dumps(category_patterns[:60], ensure_ascii=False)}
- non-overall examples:
  {json.dumps(non_overall_examples, ensure_ascii=False)}

=== PRIORITY ===
- First check for "오분류" (sentiment/category mismatch).
- Then check if the memo fits any fixed topic.
- Only use "기타" if genuinely no topic fits.
- Do NOT use "기타" as a lazy fallback. Most memos should fit a topic.

Return only JSON:
{{
  "results": [
    {{
      "row_id": "",
      "topic": "",
      "explanation": ""
    }}
  ]
}}
"""
    user = (
        "Fixed topics (with description and example memos):\n"
        + json.dumps(topic_info, ensure_ascii=False)
        + "\n\nSpecial topics: \"기타\" (no topic fits), \"오분류\" (misclassified memo)\n"
        + "\n\nMemos:\n"
        + json.dumps(
            [{"row_id": str(r["_row_id"]), "memo": clean_text(r["memo"])} for r in batch_rows],
            ensure_ascii=False,
        )
    )
    return [
        {"role": "system", "content": clean_text(system)},
        {"role": "user", "content": clean_text(user)},
    ]


print("[OK] build_classify_messages 재정의 완료")
print("     추가 특수 토픽: '기타' (분류 불가), '오분류' (감성/속성 불일치)")
print(f"     현재 그룹 감성: sc_measurement 기반 판별")
print(f"     오분류 조건: 감성 불일치 + 속성 불일치")

In [0]:
# ============================================================
# 7-1) 기존 테이블 복원: "기타(...)" 라벨 → 원래 LLM 토픽으로 복원
#      - description LIKE '희소 주제 묶음: %' → 원래 topic 추출
#      - round3 테이블 + integration 테이블 모두 복원
# ============================================================

RESTORE_TABLES = [
    DETAIL_TABLE,          # category_topic_detail_v2_round3
    V1_DETAIL_TABLE,       # category_topic_detail_integration
]

for tbl in RESTORE_TABLES:
    if not spark.catalog.tableExists(tbl):
        print(f"[SKIP] {tbl} 존재하지 않음")
        continue

    # 복원 대상 건수 확인
    restore_count = spark.sql(f"""
        SELECT COUNT(*) as cnt FROM {tbl}
        WHERE description LIKE '희소 주제 묶음: %'
    """).collect()[0]["cnt"]

    if restore_count == 0:
        print(f"[SKIP] {tbl} - 복원 대상 없음")
        continue

    print(f"[RESTORE] {tbl} - {restore_count:,}건 복원 중...")

    # topic = description에서 '희소 주제 묶음: ' 제거한 값
    spark.sql(f"""
        UPDATE {tbl}
        SET topic = substr(description, length('희소 주제 묶음: ') + 1)
        WHERE description LIKE '희소 주제 묶음: %'
    """)
    print(f"  [DONE] {restore_count:,}건 원래 토픽으로 복원 완료")

print(f"\n[ALL DONE] 기타 병합 토픽 복원 완료")

In [0]:
# ============================================================
# 7-2) Round 테이블에 catalog_v2 조인 후 integration에 합치기
#      1. DETAIL_TABLE(round) + catalog_v2 조인 → topic_rev, main_topic 추가
#      2. 전반적 긍정/부정 NULL 매핑
#      3. llm_round 컬럼 추가 (r{SAMPLING_ROUND})
#      4. 보강된 round 데이터를 integration 테이블에 머지
#      5. (v4) memo_summary 컬럼은 DETAIL_TABLE에만 유지,
#         integration 에는 미포함
# ============================================================

INTEGRATION_TABLE = "sandbox.z_jungryo_lee.category_topic_detail_integration"
CATALOG_V2_TABLE = "sandbox.z_jungryo_lee.category_topic_catalog_v2"

print(f"[STEP 1] {DETAIL_TABLE} + {CATALOG_V2_TABLE} 조인")

# 1. Round DETAIL_TABLE 로드
round_df = spark.table(DETAIL_TABLE)

# 기존 topic_rev, main_topic, llm_round 컬럼이 있으면 제거 (재실행 시 중복 방지)
for col in ["topic_rev", "main_topic", "llm_round"]:
    if col in round_df.columns:
        round_df = round_df.drop(col)

# 2. catalog_v2 조인용 DF
catalog_df = (
    spark.table(CATALOG_V2_TABLE)
    .select("cate_1_depth", "cate_2_depth", "sc_measurement", "topic", "topic_rev", "main_topic")
    .dropDuplicates(["cate_1_depth", "cate_2_depth", "sc_measurement", "topic"])
)

# 3. LEFT JOIN
enriched_df = round_df.join(
    catalog_df,
    on=["cate_1_depth", "cate_2_depth", "sc_measurement", "topic"],
    how="left"
)

# 4. 전반적 긍정/부정 토픽: NULL인 경우 topic 값으로 매핑
enriched_df = enriched_df.withColumn(
    "topic_rev",
    F.when(
        F.col("topic_rev").isNull() & F.col("topic").isin("전반적 긍정", "전반적 부정"),
        F.col("topic")
    ).otherwise(F.col("topic_rev"))
).withColumn(
    "main_topic",
    F.when(
        F.col("main_topic").isNull() & F.col("topic").isin("전반적 긍정", "전반적 부정"),
        F.col("topic")
    ).otherwise(F.col("main_topic"))
)

# 5. llm_round 컬럼 추가
round_label = f"r{SAMPLING_ROUND}"
enriched_df = enriched_df.withColumn("llm_round", F.lit(round_label))

round_count = enriched_df.count()
null_count = enriched_df.where(F.col("topic_rev").isNull()).count()
print(f"  round rows: {round_count:,}")
print(f"  topic_rev NULL: {null_count:,}")
print(f"  llm_round: {round_label}")

# 6. 보강된 round 데이터를 DETAIL_TABLE에 덮어쓰기 저장 (memo_summary 포함)
enriched_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(DETAIL_TABLE)
print(f"  [SAVE] {DETAIL_TABLE} 저장 완료 (topic_rev, main_topic, llm_round, memo_summary 포함)")

# ============================================================
# STEP 2: Integration 테이블에 머지
#   (v4) memo_summary 컬럼 제외 - 기존 포맷 유지
# ============================================================
print(f"\n[STEP 2] {INTEGRATION_TABLE}에 머지 (memo_summary 제외)")

# memo_summary 컬럼 제거용 DF
enriched_for_integration = enriched_df.drop("memo_summary") if "memo_summary" in enriched_df.columns else enriched_df

if spark.catalog.tableExists(INTEGRATION_TABLE):
    # 기존 integration에서 필요한 컬럼 없으면 추가 (schema 맞춤)
    integ_df = spark.table(INTEGRATION_TABLE)
    for col in ["topic_rev", "main_topic", "llm_round"]:
        if col not in integ_df.columns:
            spark.sql(f"ALTER TABLE {INTEGRATION_TABLE} ADD COLUMN {col} STRING")
            print(f"  [컬럼 추가] {col}")

    v1_before = spark.table(INTEGRATION_TABLE).count()
    print(f"  integration 기존: {v1_before:,}건")

    # 기존에 없는 것만 추가 (left_anti)
    existing_v1 = spark.sql(f"""
        SELECT DISTINCT memo, cate_1_depth, cate_2_depth, sc_measurement
        FROM {INTEGRATION_TABLE}
    """)

    new_rows = enriched_for_integration.join(
        existing_v1,
        on=["memo", "cate_1_depth", "cate_2_depth", "sc_measurement"],
        how="left_anti"
    )
    new_count = new_rows.count()
    print(f"  [신규 메모] {new_count:,}건 (중복 제외)")

    if new_count > 0:
        # 칼럼 순서 맞춤
        v1_columns = spark.table(INTEGRATION_TABLE).columns
        for col_name in v1_columns:
            if col_name not in new_rows.columns:
                new_rows = new_rows.withColumn(col_name, F.lit(None).cast("string"))
        new_rows_aligned = new_rows.select(*v1_columns)
        new_rows_aligned.write.mode("append").format("delta").saveAsTable(INTEGRATION_TABLE)
        print(f"  [APPEND] {new_count:,}건 추가 완료 (llm_round={round_label})")
    else:
        print(f"  [SKIP] 추가할 신규 메모 없음")

    v1_after = spark.table(INTEGRATION_TABLE).count()
    print(f"\n  integration: {v1_before:,} -> {v1_after:,} (+{v1_after - v1_before:,})")
else:
    # integration 테이블 없으면 새로 생성 (memo_summary 제외)
    enriched_for_integration.write.mode("overwrite").format("delta").saveAsTable(INTEGRATION_TABLE)
    print(f"  [CREATE] {INTEGRATION_TABLE} 생성 완료 ({enriched_for_integration.count():,}건)")

print(f"\n[ALL DONE] Round{SAMPLING_ROUND} catalog 조인 + integration 머지 완료 (메모 요약 컬럼 제외)")

In [0]:
# ============================================================
# 8) Rebuild Summary Table
# ============================================================

if not spark.catalog.tableExists(DETAIL_TABLE):
    print(f"[SKIP] {DETAIL_TABLE} 테이블이 아직 존재하지 않습니다. Cell 8 분류 파이프라인 완료 후 실행하세요.")
else:
    detail_df = spark.table(DETAIL_TABLE)

    summary_df = (
        detail_df
        .groupBy("cate_1_depth", "cate_2_depth", "sc_measurement", "topic")
        .agg(F.count("*").alias("review_count"))
        .withColumn(
            "group_total_count",
            F.sum("review_count").over(
                Window.partitionBy("cate_1_depth", "cate_2_depth", "sc_measurement")
            ),
        )
        .withColumn("review_share", F.round(F.col("review_count") / F.col("group_total_count"), 4))
        .withColumn("review_share_pct", F.round(F.col("review_share") * 100, 2))
        .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement", F.desc("review_count"))
    )

    summary_df.write.mode("overwrite").format("delta").saveAsTable(SUMMARY_TABLE)

    print(f"[SUMMARY] saved to {SUMMARY_TABLE}")
    display(summary_df)

In [0]:
# ============================================================
# 9) Version Tracking Table (버전 추적 테이블)
#    - 파이프라인 버전별 변경사항 기록
#    - v4부터 200자 초과 메모 요약 후 주제분류 적용
# ============================================================

from datetime import datetime

# 테이블 생성 (DDL)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {VERSION_TRACKING_TABLE} (
        version STRING COMMENT '버전 식별자 (e.g. v1, v2, v3, v4)',
        round_number INT COMMENT '샘플링 회차',
        change_description STRING COMMENT '변경사항 설명',
        change_detail STRING COMMENT '상세 변경 내용',
        prompt_version STRING COMMENT '프롬프트 버전',
        endpoint STRING COMMENT 'LLM 엔드포인트',
        detail_table STRING COMMENT '해당 회차 DETAIL 테이블명',
        created_at TIMESTAMP COMMENT '기록 시각',
        created_by STRING COMMENT '작성자'
    )
    USING DELTA
    COMMENT '주제분류 파이프라인 버전 추적 테이블'
""")

# 기존 버전 기록 존재 여부 확인
existing_versions = spark.sql(f"""
    SELECT version, round_number FROM {VERSION_TRACKING_TABLE}
""").collect()
existing_keys = {(r["version"], r["round_number"]) for r in existing_versions}

# 버전 로그 데이터
version_records = [
    {
        "version": "v1",
        "round_number": 1,
        "change_description": "초기 LLM 분류 파이프라인",
        "change_detail": "3-Phase 분류: 초기분류 → Overall 재분류 → 희소주제 병합. 원본 메모 그대로 분류.",
        "prompt_version": "_v2",
        "endpoint": "databricks-gpt-5-4-mini",
        "detail_table": f"{SAVE_DB}.category_topic_detail_v2",
    },
    {
        "version": "v2",
        "round_number": 2,
        "change_description": "회차별 분리 저장 + 이전 회차 중복 제거",
        "change_detail": "Round별 DETAIL 테이블 분리 저장. 이전 회차 + v1 기분류 메모 제외 후 샘플링.",
        "prompt_version": "_v2",
        "endpoint": "databricks-gpt-5-4-mini",
        "detail_table": f"{SAVE_DB}.category_topic_detail_v2_round2",
    },
    {
        "version": "v3",
        "round_number": 3,
        "change_description": "Phase 3 기타 병합 제거 + 토픽 복원",
        "change_detail": "LLM 선정 토픽 그대로 유지 (희소 주제 병합 안 함). 기존 '기타(...)' 라벨 원래 토픽으로 복원.",
        "prompt_version": "_v2",
        "endpoint": "databricks-gpt-5-4-mini",
        "detail_table": f"{SAVE_DB}.category_topic_detail_v2_round3",
    },
    {
        "version": "v4",
        "round_number": 4,
        "change_description": "200자 초과 메모 요약 후 주제분류",
        "change_detail": (
            "200자 초과 메모는 LLM으로 1문장 요약 후 요약본으로 주제분류 수행. "
            "DETAIL_TABLE에 memo_summary 컬럼 추가 (요약된 메모 저장). "
            "Integration 테이블에는 기존 컬럼 포맷 유지 (memo_summary 미포함). "
            f"MEMO_SUMMARIZE_THRESHOLD={MEMO_SUMMARIZE_THRESHOLD}, SUMMARIZE_BATCH_SIZE={SUMMARIZE_BATCH_SIZE}"
        ),
        "prompt_version": "_v2",
        "endpoint": ENDPOINT,
        "detail_table": DETAIL_TABLE,
    },
]

# 신규 버전만 INSERT
new_records = []
for rec in version_records:
    key = (rec["version"], rec["round_number"])
    if key not in existing_keys:
        new_records.append(rec)

if new_records:
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
    
    schema = StructType([
        StructField("version", StringType(), True),
        StructField("round_number", IntegerType(), True),
        StructField("change_description", StringType(), True),
        StructField("change_detail", StringType(), True),
        StructField("prompt_version", StringType(), True),
        StructField("endpoint", StringType(), True),
        StructField("detail_table", StringType(), True),
        StructField("created_at", TimestampType(), True),
        StructField("created_by", StringType(), True),
    ])
    
    rows = [
        {
            **rec,
            "created_at": datetime.now(),
            "created_by": "jungryo.lee@lge.com",
        }
        for rec in new_records
    ]
    version_df = spark.createDataFrame(rows, schema=schema)
    append_table(version_df, VERSION_TRACKING_TABLE)
    print(f"[VERSION LOG] {len(new_records)}건 신규 버전 기록 추가")
else:
    print("[VERSION LOG] 이미 모든 버전 기록 존재")

print(f"\n[VERSION TRACKING TABLE] {VERSION_TRACKING_TABLE}")
display(spark.table(VERSION_TRACKING_TABLE).orderBy("round_number"))

In [0]:
# ============================================================
# 10) 최종 결과 적재: voc_llm_topic_classification
#     - 해당 회차 결과를 data_created_dt 추가하여 적재
#     - 대상 테이블: sandbox.t_online_voc_analysis.voc_llm_topic_classification
#     - 적재 방식: append (회차별 누적)
# ============================================================

from pyspark.sql import functions as F

FINAL_OUTPUT_TABLE = "sandbox.t_online_voc_analysis.voc_llm_topic_classification"

print(f"[STEP] 회차 {SAMPLING_ROUND} 결과 \u2192 {FINAL_OUTPUT_TABLE} 적재")
print(f"  소스: {DETAIL_TABLE}")

# 1. 현재 회차 DETAIL_TABLE 로드 (catalog 조인 완료된 상태)
if not spark.catalog.tableExists(DETAIL_TABLE):
    raise RuntimeError(f"[ERROR] {DETAIL_TABLE} 존재하지 않음. Cell 7-2 실행 후 진행하세요.")

round_df = spark.table(DETAIL_TABLE)
print(f"  회차 데이터: {round_df.count():,}건")

# 2. data_created_dt 컨럼 추가 (DATE 타입 - 기존 테이블 스키마 일치)
output_df = round_df.withColumn("data_created_dt", F.current_date())

# 3. 최종 적재 컬럼 선택 (필요 컬럼만 정리)
#    - memo_summary는 제외 (운영 테이블에는 불필요)

final_columns = [
    "cate_1_depth", "cate_2_depth", "sc_measurement",
    "topic", "_row_id", "memo",  "description",
    "topic_rev", "main_topic", "llm_round",
    "data_created_dt"
]

# 실제 존재하는 컬럼만 선택
available_cols = [c for c in final_columns if c in output_df.columns]
output_df = output_df.select(*available_cols)

print(f"  적재 컬럼: {available_cols}")
print(f"  적재 건수: {output_df.count():,}")

# 4. 테이블 존재 여부에 따라 append / create
if spark.catalog.tableExists(FINAL_OUTPUT_TABLE):
    # 기존 테이블 존재 \u2192 중복 방지 후 append
    existing_df = spark.table(FINAL_OUTPUT_TABLE)
    existing_count = existing_df.count()
    print(f"  기존 테이블: {existing_count:,}건")
    
    # 해당 회차 데이터가 이미 있으면 중복 제거
    round_label = f"r{SAMPLING_ROUND}"
    already_loaded = existing_df.where(F.col("llm_round") == round_label).count()
    
    if already_loaded > 0:
        print(f"  \u26a0\ufe0f 회차 {round_label} 데이터 {already_loaded:,}건 이미 존재 \u2192 삭제 후 재적재")
        spark.sql(f"DELETE FROM {FINAL_OUTPUT_TABLE} WHERE llm_round = '{round_label}'")
    
    output_df.write.mode("append").format("delta").saveAsTable(FINAL_OUTPUT_TABLE)
    final_count = spark.table(FINAL_OUTPUT_TABLE).count()
    print(f"  [APPEND] {existing_count:,} \u2192 {final_count:,} (+{final_count - existing_count + already_loaded:,})")
else:
    # 신규 생성
    output_df.write.mode("overwrite").format("delta").saveAsTable(FINAL_OUTPUT_TABLE)
    print(f"  [CREATE] {FINAL_OUTPUT_TABLE} 생성 ({output_df.count():,}건)")

print(f"\n\u2705 적재 완료: {FINAL_OUTPUT_TABLE}")
print(f"   data_created_dt = current_date()")
print(f"   llm_round = r{SAMPLING_ROUND}")

# 검증
display(
    spark.table(FINAL_OUTPUT_TABLE)
    .groupBy("llm_round")
    .agg(F.count("*").alias("row_count"), F.min("data_created_dt").alias("min_dt"), F.max("data_created_dt").alias("max_dt"))
    .orderBy("llm_round")
)